# Vehicle Image Classification using CNN
## Car • Motorbike (Bicycle dataset will be added later)

This notebook builds a Convolutional Neural Network (CNN) to classify vehicle images into two categories for now.

**Before running:** add labelled images to `Dataset/train`, `Dataset/validation`, and `Dataset/test`. See `README.md` for the folder structure and suggested dataset split.


## 1. Import libraries
Import TensorFlow for the CNN, Matplotlib for graphs, and helper libraries for paths and files.

In [ ]:
from pathlib import Path
import json
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)


## 2. Set project settings
All images are resized to 200 × 200 pixels. The class order is fixed so that prediction labels stay consistent.

In [ ]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "Dataset"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

IMG_SIZE = (200, 200)
BATCH_SIZE = 32
EPOCHS = 15
SEED = 42
CLASS_NAMES = ["Car", "Motorbike"]

print("Dataset folder:", DATA_DIR)


## 3. Check dataset folders
Each split must contain both class folders, and each class folder must contain images.

In [ ]:
extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for split in ["train", "validation", "test"]:
    for class_name in CLASS_NAMES:
        folder = DATA_DIR / split / class_name
        image_count = sum(1 for p in folder.rglob("*")
                          if p.suffix.lower() in extensions)
        print(f"{split}/{class_name}: {image_count} images")
        if image_count == 0:
            raise ValueError(
                f"No images found in {folder}. Add labelled images first."
            )


## 4. Load the images
`image_dataset_from_directory` reads images and assigns labels from their folder names.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=CLASS_NAMES,
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=CLASS_NAMES,
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    class_names=CLASS_NAMES,
    shuffle=False,
)


## 5. Display sample training images
This helps check whether images have been placed in the correct folders.

In [ ]:
plt.figure(figsize=(9, 7))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(CLASS_NAMES[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()


## 6. Prepare the input pipeline
Caching and prefetching can make training more efficient.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(500).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


## 7. Build the CNN model
Convolution and pooling layers learn visual features. The final Softmax layer produces probabilities for the two vehicle categories for now.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
])

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(len(CLASS_NAMES), activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## 8. Train the model
The model learns from training images. Validation data is used to monitor performance after each epoch.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_DIR / "vehicle_cnn.keras",
        monitor="val_accuracy",
        save_best_only=True,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)


## 9. Plot accuracy and loss
These graphs show how training and validation performance change over epochs.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("CNN Accuracy")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("CNN Loss")
plt.grid(True)
plt.show()


## 10. Evaluate using test images
The test dataset is kept separate from training and validation. Its result gives a final evaluation on unseen images.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test accuracy: {test_accuracy * 100:.2f}%")
print(f"Test loss: {test_loss:.4f}")


## 11. Save the model and class names
The saved model can be reused later without training again.

In [ ]:
model.save(MODEL_DIR / "vehicle_cnn.keras")
(MODEL_DIR / "class_names.json").write_text(
    json.dumps(CLASS_NAMES, indent=2),
    encoding="utf-8"
)
print("Model and class names saved.")


## 12. Predict one new image
Change `IMAGE_PATH` to the location of a vehicle image on your computer. The image is resized and passed to the trained model.

In [ ]:
IMAGE_PATH = Path("sample_vehicle.jpg")  # Change this path

if not IMAGE_PATH.is_file():
    print("Please set IMAGE_PATH to an existing car, motorbike, or bicycle image.")
else:
    image = tf.keras.utils.load_img(IMAGE_PATH, target_size=IMG_SIZE)
    image_array = tf.keras.utils.img_to_array(image)
    image_array = tf.expand_dims(image_array, 0)

    probabilities = model.predict(image_array, verbose=0)[0]
    best_index = int(tf.argmax(probabilities))

    print("Prediction:", CLASS_NAMES[best_index])
    print(f"Confidence: {probabilities[best_index] * 100:.2f}%")

    for name, score in zip(CLASS_NAMES, probabilities):
        print(f"{name}: {score * 100:.2f}%")


## 13. Conclusion

This project demonstrates how a CNN can learn image features and classify a vehicle image into **Car**, **Motorbike**, or **Bicycle**.

The final accuracy depends on the quality, quantity, balance, and variety of the dataset. Record the actual test accuracy after running the notebook; do not invent a result.
